# SQ-2 — Per-subject metadata fidelity

**Tests:** does the FLTA rule engine include and exclude data subjects correctly on the basis of their pod-resident metadata?

Pod resources exercised:
- `consent.jsonld` — Consent Receipt (purposes, lawful basis, validity, signature)
- `withdrawal.jsonld` — withdrawal log
- `jurisdiction.jsonld` — subject / controller / operator jurisdictions
- `participation.jsonld` — FL participation log; SOLID-LIFECYCLE-001 fires when a round is timestamped after a withdrawal

**Success criteria** ([METHODOLOGY.md](../../METHODOLOGY.md) §1):
- 0 false inclusions on the labelled federation
- 0 false exclusions
- Exact firing-set match against `pods/_manifest.json` expected entries

In [1]:
import json, sys, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

EVAL_DIR = Path.cwd().parents[1] if Path.cwd().name.startswith('sq') else Path.cwd()
sys.path.insert(0, str(EVAL_DIR))

from flta_eval import audit, rules
from flta_eval import pods as podsmod

NOW = datetime(2026, 5, 25, 12, 0, 0, tzinfo=timezone.utc)
card = json.loads((EVAL_DIR / 'card' / 'example_chain.json').read_text())
manifest = json.loads((EVAL_DIR / 'pods' / '_manifest.json').read_text())
print(f'federation size: {manifest["size"]} (positive {manifest["positive_count"]}, negative {manifest["negative_count"]})')

federation size: 200 (positive 100, negative 100)


In [2]:
t0 = time.time()
projections = []
for entry in manifest['entries']:
    pod = podsmod.load_pod(EVAL_DIR / 'pods' / entry['subject_id'])
    proj = rules.project_residual_for_subject(card, pod, now=NOW)
    projections.append({
        'subject_id': entry['subject_id'],
        'expected_inclusion': entry['expected_inclusion'],
        'actual_inclusion': proj['included'],
        'expected_firings': sorted(entry['expected_firings']),
        'actual_firings': sorted(set(proj['firings'])),
    })
elapsed = time.time() - t0
fi = [p for p in projections if not p['expected_inclusion'] and p['actual_inclusion']]
fe = [p for p in projections if p['expected_inclusion'] and not p['actual_inclusion']]
firings_match = [p for p in projections if p['expected_firings'] == p['actual_firings']]
print(f'elapsed: {elapsed:.2f}s ({1000*elapsed/len(projections):.2f} ms/pod)')
print()
print(f'false inclusions      : {len(fi)} / {len(projections)}')
print(f'false exclusions      : {len(fe)} / {len(projections)}')
print(f'firing-set exact match: {len(firings_match)} / {len(projections)}')

elapsed: 0.33s (1.65 ms/pod)

false inclusions      : 0 / 200
false exclusions      : 0 / 200
firing-set exact match: 200 / 200


In [3]:
by_rule_expected = Counter()
by_rule_correct = Counter()
for p in projections:
    for r in p['expected_firings']:
        by_rule_expected[r] += 1
        if r in p['actual_firings']:
            by_rule_correct[r] += 1

print(f"{'rule':25s} {'expected':>10s} {'fired':>8s} {'precision':>11s}")
for rule_id in sorted(by_rule_expected):
    exp = by_rule_expected[rule_id]
    cor = by_rule_correct[rule_id]
    precision = cor / exp if exp else 1.0
    print(f"  {rule_id:23s} {exp:>10d} {cor:>8d} {precision:>10.2%}")

rule                        expected    fired   precision
  SOLID-CONSENT-CONTROLLER         10       10    100.00%
  SOLID-CONSENT-EXPIRED           25       25    100.00%
  SOLID-CONSENT-SIGNATURE         15       15    100.00%
  SOLID-JURIS-001                 15       15    100.00%
  SOLID-LIFECYCLE-001             15       15    100.00%
  SOLID-WITHDRAW-001              35       35    100.00%


In [4]:
record_path = audit.write_result_record(
    target_dir=EVAL_DIR / 'results' / 'sq2',
    attack='per_subject_fidelity',
    variant='SQ-2',
    threat_profile='R',
    dataset={'name': 'pod_federation_synthetic_personae_v2',
             'n': len(projections),
             'positive': manifest['positive_count'],
             'negative': manifest['negative_count']},
    config={'card_id': card['card_id'], 'evaluation_now': NOW.isoformat()},
    seed_namespace='sq2.per_subject_fidelity.v2',
    result={
        'false_inclusions': len(fi),
        'false_exclusions': len(fe),
        'firing_set_exact_matches': len(firings_match),
        'total': len(projections),
        'per_rule_precision': {r: {'expected': by_rule_expected[r],
                                    'fired': by_rule_correct[r],
                                    'precision': by_rule_correct[r]/by_rule_expected[r]}
                               for r in by_rule_expected},
    },
    tolerances={'false_inclusions': 0, 'false_exclusions': 0},
)
print(f'wrote {record_path.relative_to(EVAL_DIR)}')

wrote results/sq2/per_subject_fidelity__SQ-2__2026-05-26T08-56-31Z.json


In [5]:
# Inspect one positive pod and one negative pod
import textwrap
def show(subject_id):
    p = EVAL_DIR / 'pods' / subject_id
    print(f'─── {subject_id} ───')
    for res in ['consent.jsonld', 'withdrawal.jsonld', 'jurisdiction.jsonld', 'participation.jsonld']:
        obj = json.loads((p / res).read_text())
        print(f'{res}:')
        print(textwrap.indent(json.dumps(obj, indent=2), '  '))
show('persona_pos_000')
print()
show('persona_neg_000')

─── persona_pos_000 ───
consent.jsonld:
  {
    "@context": "https://www.w3.org/ns/solid/consent/v1",
    "@type": "ConsentReceipt",
    "controller": "https://federation.example/controller/fl-bloodmnist-2026",
    "issued": "2026-02-05T12:00:00Z",
    "lawful_basis": "Art. 6(1)(a) GDPR \u2014 consent + Art. 9(2)(j) \u2014 research",
    "not_after": "2027-02-05T12:00:00Z",
    "purposes": [
      "research",
      "model-training"
    ],
    "signature": "sig:persona_pos_000",
    "subject_webid": "https://example.solidcommunity.net/persona_pos_000/profile/card#me",
    "withdrawable": true
  }
withdrawal.jsonld:
  {
    "@context": "https://www.w3.org/ns/solid/consent/v1",
    "@type": "WithdrawalLog",
    "entries": [],
    "subject_webid": "https://example.solidcommunity.net/persona_pos_000/profile/card#me"
  }
jurisdiction.jsonld:
  {
    "@type": "JurisdictionalTag",
    "controller_jurisdiction": "EU",
    "cross_border_mechanism": null,
    "data_subject_jurisdiction": "EU",
  